# 03 — Preparación de los datos (Data Preparation)

**Fase CRISP-DM:** Preparación de los datos.

**Lee de:** `data/raw/` (arabica y robusta, combinados).

**Produce:** `data/processed/coffee_processed.csv` (dataset completo, limpio y con target), `data/processed/train.csv` y `data/processed/test.csv` (split estratificado).

Toda la lógica de esta fase vive en `src/` (arquitectura por capas): `src/data/clean_data.py` (limpieza), `src/features/build_features.py` (target + preprocesamiento) y `src/pipeline.py` (orquestación). Este notebook solo llama a `pipeline.prepare_data()` y documenta las decisiones tomadas.

## Decisiones tomadas en esta fase

- **Registro corrupto (`Total.Cup.Points == 0`):** se elimina antes de calcular el target (`src/data/clean_data.py::DropCorruptScores`), ver hallazgo de 02.
- **Variable objetivo:** `especialidad = 1` si `Total.Cup.Points >= 80` (umbral SCA), `0` en caso contrario (`src/features/build_features.py::SpecialtyTargetBuilder`), configurado en `config/config.yaml -> target`.
- **Arabica + Robusta:** se combinan en un solo dataset con columna `species`; el desbalance de la variable objetivo se trata con SMOTE en la fase de modelado (04), no aquí.
- **Variables predictoras:** solo variables de contexto — `Number.of.Bags`, `altitude_mean_meters`, `Moisture`, `Category.One.Defects`, `Category.Two.Defects`, `Quakers` (numéricas) y `Country.of.Origin`, `Variety`, `Processing.Method`, `Color`, `species` (categóricas). Se excluyen deliberadamente los subpuntajes de catación (`Aroma`, `Flavor`, ...) porque son los componentes que suman `Total.Cup.Points`, y usarlos filtraría directamente el target (ver 02.4).
- **Nulos:** columnas numéricas → mediana; columnas categóricas → categoría `"Unknown"` (`SimpleImputer`, dentro del `ColumnTransformer` de `src/features/build_features.py::build_preprocessor`). No se descartan filas ni columnas por nulos.
- **Encoding/escalado:** `OneHotEncoder` para categóricas, `StandardScaler` para numéricas.
- **Split:** 80/20 estratificado por `especialidad`, semilla `config.yaml -> project.random_seed` (42).

In [1]:
import sys
sys.path.insert(0, "..")  # para poder importar src/ desde notebooks/

from src.config import load_config
from src import pipeline

config = load_config()
result = pipeline.prepare_data(config)

print("Filas duplicadas eliminadas:", result["n_duplicates_dropped"])
print("Filas con Total.Cup.Points corrupto eliminadas:", result["n_corrupt_dropped"])
print("Shape dataset procesado:", result["processed"].shape)

Filas duplicadas eliminadas: 0
Filas con Total.Cup.Points corrupto eliminadas: 1
Shape dataset procesado: (1338, 51)


In [2]:
print("Balance de la variable objetivo (dataset completo):")
print(result["processed"][config["target"]["target_column"]].value_counts(normalize=True).round(3))

print("\nShapes de los splits:")
print("X_train:", result["X_train"].shape, " X_test:", result["X_test"].shape)
print("\nBalance en train:")
print(result["y_train"].value_counts(normalize=True).round(3))
print("\nBalance en test:")
print(result["y_test"].value_counts(normalize=True).round(3))

Balance de la variable objetivo (dataset completo):
especialidad
1    0.861
0    0.139
Name: proportion, dtype: float64

Shapes de los splits:
X_train: (1070, 11)  X_test: (268, 11)

Balance en train:
especialidad
1    0.861
0    0.139
Name: proportion, dtype: float64

Balance en test:
especialidad
1    0.862
0    0.138
Name: proportion, dtype: float64


**Hallazgo:** con el umbral SCA (>= 80), ~86% de los lotes califican como "café de especialidad" y ~14% no — un desbalance real (no tan severo como el de especie, pero relevante), consistente en train y test gracias al split estratificado. Esto confirma que sí valía la pena tratar el desbalance con SMOTE en la fase 04 (04_modelado.ipynb).

Los artefactos (`coffee_processed.csv`, `train.csv`, `test.csv`) ya quedaron guardados en `data/processed/` al llamar `pipeline.prepare_data()`.